# TSC 标准 Unsloth GRPO 训练

使用标准 Unsloth GRPOTrainer + 离线 Dataset + 自定义 Reward Function

## 1. 环境配置

## 1.5 生成/检查 Dataset（可选）

如果 dataset 不存在，此 cell 会自动生成；如果已存在，则跳过。

In [1]:
import os
from pathlib import Path

DATASET_PATH = "grpo_dataset"

# 检查 dataset 是否已存在
if os.path.exists(DATASET_PATH):
    print(f"✓ Dataset 已存在: {DATASET_PATH}")
    print("跳过生成，直接使用现有 dataset")
else:
    print(f"⚠️ Dataset 不存在: {DATASET_PATH}")
    print("开始生成 dataset...\n")
    
    # 导入生成模块
    import sys
    sys.path.insert(0, os.getcwd())
    
    # 导入并运行生成函数
    from generate_grpo_dataset import main as generate_main, CONFIG
    
    # 配置生成参数（可根据需要调整）
    print("当前配置:")
    print(f"  - warmup_steps: {CONFIG['warmup_steps']}")
    print(f"  - steps_per_tl: {CONFIG['steps_per_tl']}")
    print(f"  - max_tl_per_scenario: {CONFIG['max_tl_per_scenario']}")
    print(f"  - num_workers: {CONFIG['num_workers']} (并行)")
    print()
    
    # 快速测试配置（取消注释以使用）
    CONFIG['steps_per_tl'] = 5
    CONFIG['max_tl_per_scenario'] = 2
    CONFIG['num_workers'] =10
    
    # 开始生成
    dataset = generate_main()
    print(f"\n✓ Dataset 生成完成！")

⚠️ Dataset 不存在: grpo_dataset
开始生成 dataset...

当前配置:
  - warmup_steps: 80
  - steps_per_tl: 20
  - max_tl_per_scenario: 5
  - num_workers: 4 (并行)

发现场景数: 1402
总训练组合数: 2804
  优先场景: 4
  其他场景: 2800
  并行 workers: 10

开始并行生成（10 workers）...
DEBUG: 使用的 SUMO 路径是: /usr/share/sumo/bin/sumoDEBUG: 使用的 SUMO 路径是: /usr/share/sumo/bin/sumoDEBUG: 使用的 SUMO 路径是: /usr/share/sumo/bin/sumoDEBUG: 使用的 SUMO 路径是: /usr/share/sumo/bin/sumoDEBUG: 使用的 SUMO 路径是: /usr/share/sumo/bin/sumo

DEBUG: 使用的 SUMO 路径是: /usr/share/sumo/bin/sumoDEBUG: 使用的 SUMO 路径是: /usr/share/sumo/bin/sumoDEBUG: 使用的 SUMO 路径是: /usr/share/sumo/bin/sumo
Starting SUMO with command: /usr/share/sumo/bin/sumo -c /home/samuel/unsloth_dgx/workspace/SCU_TSC/sumo_simulation/environments/cologne8/cologne8.sumocfg --step-length 1.0 --no-warnings true --start
DEBUG: 使用的 SUMO 路径是: /usr/share/sumo/bin/sumo
DEBUG: 使用的 SUMO 路径是: /usr/share/sumo/bin/sumoStarting SUMO with command: /usr/share/sumo/bin/sumo -c /home/samuel/unsloth_dgx/workspace/SCU_TSC/sumo_simulatio

Saving the dataset (0/1 shards):   0%|          | 0/14020 [00:00<?, ? examples/s]

✓ Dataset 已保存到: grpo_dataset

=== Dataset 统计 ===
总样本数: 14020
场景数: 1402
信号灯数: 6

✓ Dataset 生成完成！


In [18]:
from datasets import load_from_disk

DATASET_PATH = "grpo_dataset"

if not os.path.exists(DATASET_PATH):
    raise FileNotFoundError(
        f"Dataset 不存在: {DATASET_PATH}\n"
        "请先运行 generate_grpo_dataset.py 生成离线 dataset"
    )

dataset = load_from_disk(DATASET_PATH)
print(f"✓ 加载 Dataset: {len(dataset)} 个样本")
print(f"Dataset 列: {dataset.column_names}")
print(f"\n示例样本:")
print(dataset[2])

✓ 加载 Dataset: 14020 个样本
Dataset 列: ['prompt', 'state_path', 'scenario', 'tl_id', 'phase_order', 'phase_limits', 'step_idx']

示例样本:
{'prompt': [{'content': '你是交通信号配时优化专家。\n你将收到一个 JSON（用【cycle_predict_input_json】...【/cycle_predict_input_json】包裹）。\n你的任务是：在满足硬约束前提下，输出下一周期各相位最终绿灯时间 final（单位：秒）。\n注意：history 数据可能不完整（例如 only recent_cycles；yesterday_same_time/last_week_same_time 可能为 null），必须基于可用部分输出结果。\n只输出最终 JSON 数组(list)，不要输出任何解释/过程。\n', 'role': 'system'}, {'content': '【cycle_predict_input_json】{"crossing_id":831670537,"as_of":"2026-01-14 17:34:58","phase_order":[1,2,3,4],"phase_limits":{"1":{"min_green":6,"max_green":42},"2":{"min_green":21,"max_green":93},"3":{"min_green":7,"max_green":54},"4":{"min_green":28,"max_green":84}},"history":{"recent_cycles":[{"time":"2026-01-14 17:34:58","phase_waits":[{"phase_id":1,"avg_wait":0.0},{"phase_id":2,"avg_wait":0.0},{"phase_id":3,"avg_wait":3.5},{"phase_id":4,"avg_wait":0.0}]},{"time":"2026-01-14 17:34:58","phase_waits":[{"phase_id":1,"avg_wait":0.0}

In [ ]:
import os
os.environ["UNSLOTH_USE_MODELSCOPE"] = "1"

In [ ]:
import subprocess
result = subprocess.run('bash -c "source /etc/network_turbo && env | grep proxy"', shell=True, capture_output=True, text=True)
for line in result.stdout.splitlines():
    if '=' in line:
        var, value = line.split('=', 1)
        os.environ[var] = value

## 2. 加载模型

In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048
lora_rank = 32

os.environ["HF_HOME"] = 'model'
os.environ["MODELSCOPE_CACHE"] = 'model'

# 基础模型或 checkpoint
BASE_MODEL_DIR = "model/models/qwen3-4B-SFT"
CHECKPOINT_DIR = "checkpoints/grpo_tsc_latest"

def _looks_like_checkpoint(path: str) -> bool:
    if not os.path.isdir(path):
        return False
    marker_files = [
        "adapter_config.json",
        "adapter_model.safetensors",
        "adapter_model.bin",
        "config.json",
    ]
    return any(os.path.isfile(os.path.join(path, f)) for f in marker_files)

resume_from = CHECKPOINT_DIR if _looks_like_checkpoint(CHECKPOINT_DIR) else BASE_MODEL_DIR
if resume_from == CHECKPOINT_DIR:
    print(f"✓ 从 checkpoint 继续训练: {CHECKPOINT_DIR}")
else:
    print(f"ℹ 从基础模型开始: {BASE_MODEL_DIR}")

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=resume_from,
    max_seq_length=max_seq_length,
    load_in_4bit=False,
    fast_inference=False,
    max_lora_rank=lora_rank,
    gpu_memory_utilization=0.8,
)

if resume_from == BASE_MODEL_DIR:
    model = FastLanguageModel.get_peft_model(
        model,
        r=lora_rank,
        target_modules=[
            "q_proj", "k_proj", "v_proj", "o_proj",
            "gate_proj", "up_proj", "down_proj",
        ],
        lora_alpha=lora_rank * 2,
        use_gradient_checkpointing="unsloth",
        random_state=3407,
    )
else:
    try:
        model.gradient_checkpointing_enable()
    except Exception:
        pass
    _trainable = [p for p in model.parameters() if p.requires_grad]
    if len(_trainable) == 0:
        for name, p in model.named_parameters():
            if "lora" in name.lower():
                p.requires_grad = True
        print("⚠️ checkpoint 未检测到可训练参数，已强制启用 LoRA 参数训练")

## 3. 加载 Dataset

## 4. 导入 Reward Function

In [ ]:
from tsc_reward_function import tsc_reward_fn, cleanup_global_pool

print("✓ Reward function 加载成功")

## 5. 配置 GRPOTrainer

In [ ]:
from trl import GRPOConfig, GRPOTrainer

config = GRPOConfig(
    output_dir="checkpoints/grpo_tsc",
    
    # 批次配置
    per_device_train_batch_size=2,       # 每 GPU 每次取多少个 prompt
    num_generations=8,                   # 每个 prompt 生成多少个 completion
    gradient_accumulation_steps=4,       # 梯度累积步数
    
    # 生成配置
    max_completion_length=256,
    temperature=0.8,                     # 非零温度自动启用采样
    top_p=0.95,
    top_k=50,
    
    # 训练配置
    learning_rate=2e-6,
    num_train_epochs=1,
    max_steps=-1,                        # -1 表示用 num_train_epochs
    
    # GRPO 特定
    scale_rewards=True,                  # 按 group 标准化 reward
    
    # 日志与保存
    logging_steps=5,
    save_steps=100,
    save_total_limit=3,
    
    # 优化器
    optim="adamw_torch",
    weight_decay=0.01,
    warmup_steps=50,
    
    # 其他
    bf16=True,
    report_to="none",                    # 或 "wandb" 如果需要 W&B
    remove_unused_columns=False,         # 保留 dataset 额外字段
)

print("✓ GRPOConfig 配置完成")

## 6. 创建 GRPOTrainer 并开始训练

In [ ]:
trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    args=config,
    train_dataset=dataset,
    reward_funcs=tsc_reward_fn,          # 自定义 reward 函数
)

print("✓ GRPOTrainer 创建成功")
print(f"训练样本数: {len(dataset)}")
print(f"每 epoch steps: {len(trainer.get_train_dataloader())}")

# 开始训练
print("\n" + "="*60)
print("开始 GRPO 训练")
print("="*60 + "\n")

trainer.train()

print("\n" + "="*60)
print("训练完成")
print("="*60)

## 7. 保存最终模型

In [ ]:
# 保存最终 checkpoint
final_output_dir = "checkpoints/grpo_tsc_final"
trainer.save_model(final_output_dir)
tokenizer.save_pretrained(final_output_dir)
print(f"✓ 最终模型已保存到: {final_output_dir}")

## 8. 清理资源

In [ ]:
# 清理 SUMO simulator 池
cleanup_global_pool()
print("✓ Simulator 池已清理")

## 9. 测试推理（可选）

In [ ]:
# 测试推理
FastLanguageModel.for_inference(model)

# 使用 dataset 中的一个样本作为测试
test_sample = dataset[0]
test_prompt = test_sample['prompt']

# 应用 chat template
prompt_text = tokenizer.apply_chat_template(test_prompt, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(prompt_text, return_tensors="pt").to(model.device)

print("测试生成:")
print("-" * 60)

outputs = model.generate(
    **inputs,
    max_new_tokens=256,
    temperature=0.7,
    top_p=0.9,
    do_sample=True,
)

generated_text = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
print(generated_text)
print("-" * 60)